## Download Libraries

In [ ]:
!pip install docx2txt
!pip install PyPDF2
!pip install dateparser
import re
import docx2txt
import nltk
import pandas as pd
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer, util
import spacy
import json
import dateparser
from datetime import datetime
import logging
from typing import Dict, List, Optional, Tuple
import warnings

warnings.filterwarnings('ignore')

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("averaged_perceptron_tagger")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 7.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

Input Handling

In [ ]:
def extract_text(file_path: str) -> str:
    """Extract text from PDF, DOCX, TXT files"""
    try:
        if file_path.endswith('.pdf'):
            reader = PdfReader(file_path)
            text = ''
            for page in reader.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + '\n'
            return text
        elif file_path.endswith('.docx'):
            return docx2txt.process(file_path)
        elif file_path.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                return f.read()
        else:
            raise ValueError(f"Unsupported file format: {file_path}")
    except Exception as e:
        logger.error(f"Error extracting text from {file_path}: {str(e)}")
        raise

Text Cleaning & Tokenization

In [ ]:
def clean_text(text: str) -> str:
    """Clean text while preserving important characters"""
    text = text.replace('\n', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def split_sentences(text: str) -> List[str]:
    """Split text into sentences"""
    sentences = nltk.sent_tokenize(text)
    return sentences

Load Models (With Caching)

In [ ]:
logger.info("Loading models...")
try:
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
    nlp = spacy.load("en_core_web_sm")
    logger.info("Models loaded successfully!")
except Exception as e:
    logger.error(f"Error loading models: {str(e)}")
    raise

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Regex Patterns

In [ ]:
email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
phone_pattern = r'(?<!/)(?<!\d)\+?[\d\s()-]{10,}(?!\d)'
linkedin_pattern = r'(?:https?://)?(?:www\.)?linkedin\.com/in/[\w\-]+'
github_pattern = r'(?:https?://)?(?:www\.)?github\.com/[\w\-]+'

# Enhanced date patterns
month_pattern = r'(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember))'
year_pattern = r'\d{4}'
date_pattern = f'(?:{month_pattern}\\s+{year_pattern}|{year_pattern})'
duration_pattern = f'({date_pattern})\\s*[-–—to]+\\s*(Present|Current|Now|{date_pattern})'

Upload Files

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving job_titles.csv to job_titles.csv
Saving ShahdRaafat.pdf to ShahdRaafat.pdf
Saving Aya_Asfour__CV.pdf to Aya_Asfour__CV.pdf
Saving C.V_Youssef_Ashraf[1].pdf to C.V_Youssef_Ashraf[1].pdf
Saving Amr Muarak CV.pdf to Amr Muarak CV.pdf
Saving cv.pdf to cv.pdf
Saving skills.csv to skills.csv


Load Skills and Job Titles CSV

In [ ]:
def load_reference_data():
    """Load skills and job titles from CSV files with fallback"""
    try:
        # Try to load skills.csv
        try:
            skills_df = pd.read_csv('skills.csv')
            if 'skill_name' not in skills_df.columns:
                logger.warning("skills.csv missing 'skill_name' column, using fallback")
                skills_list = FALLBACK_SKILLS
            else:
                skills_list = skills_df['skill_name'].dropna().str.strip().unique().tolist()
                # Remove empty strings
                skills_list = [s for s in skills_list if s]
                logger.info(f"✅ Loaded {len(skills_list)} skills from skills.csv")
        except FileNotFoundError:
            logger.warning("❌ skills.csv not found, using fallback list")
            skills_list = FALLBACK_SKILLS

        # Try to load job_titles.csv
        try:
            job_titles_df = pd.read_csv('job_titles.csv')
            if 'title_name' not in job_titles_df.columns:
                logger.warning("job_titles.csv missing 'title_name' column, using fallback")
                job_titles_list = FALLBACK_JOB_TITLES
            else:
                job_titles_list = job_titles_df['title_name'].dropna().str.strip().unique().tolist()
                job_titles_list = [t for t in job_titles_list if t]
                logger.info(f"✅ Loaded {len(job_titles_list)} job titles from job_titles.csv")
        except FileNotFoundError:
            logger.warning("❌ job_titles.csv not found, using fallback list")
            job_titles_list = FALLBACK_JOB_TITLES

        # Create embeddings
        logger.info("Creating embeddings...")
        skills_embeddings = sbert_model.encode(skills_list, show_progress_bar=False, batch_size=32)
        job_titles_embeddings = sbert_model.encode(job_titles_list, show_progress_bar=False, batch_size=32)

        return skills_list, skills_embeddings, job_titles_list, job_titles_embeddings

    except Exception as e:
        logger.error(f"Error loading reference data: {str(e)}")
        logger.info("Using fallback lists")
        skills_embeddings = sbert_model.encode(FALLBACK_SKILLS, show_progress_bar=False)
        job_titles_embeddings = sbert_model.encode(FALLBACK_JOB_TITLES, show_progress_bar=False)
        return FALLBACK_SKILLS, skills_embeddings, FALLBACK_JOB_TITLES, job_titles_embeddings

# Load data
skills_list, skills_embeddings, job_titles_list, job_titles_embeddings = load_reference_data()

Helper Functions

In [ ]:

TECH_KEYWORDS = {
    'typescript', 'javascript', 'python', 'java', 'html', 'css', 'php', 'sql',
    'react', 'next.js', 'redux', 'tailwind', 'mysql', 'postgresql', 'sqlite',
    'git', 'github', 'supabase', 'node.js', 'express', 'mongodb', 'aws',
    'docker', 'kubernetes', 'jenkins', 'c++', 'c#', 'ruby', 'swift', 'kotlin',
    'flutter', 'vue', 'angular', 'django', 'flask', 'laravel', 'spring'
}

def extract_name(text: str) -> str:
    """Extract name from first lines of CV"""
    lines = text.split('\n')[:5]

    for line in lines:
        line = line.strip()
        if 5 < len(line) < 50:
            words = line.split()
            if 2 <= len(words) <= 4 and all(w[0].isupper() for w in words if w):
                if not any(char in line for char in ['@', '+', 'http', '|', '–']):
                    return line

    # Fallback to spaCy
    doc = nlp(' '.join(lines))
    for ent in doc.ents:
        if ent.label_ == "PERSON" and len(ent.text.split()) >= 2:
            return ent.text

    return ""

def extract_contact_info(text: str) -> Dict[str, List[str]]:
    """Extract all contact information with improved patterns"""
    emails = list(set(re.findall(email_pattern, text)))

    # Phone extraction with better filtering
    phone_matches = re.findall(phone_pattern, text)
    # Filter out numbers that appear in URLs or are too short
    phones = []
    for phone in phone_matches:
        clean_phone = re.sub(r'[^\d+]', '', phone)
        if len(clean_phone) >= 10 and 'linkedin' not in text[max(0, text.find(phone)-20):text.find(phone)+20].lower():
            phones.append(phone.strip())

    return {
        'email': emails,
        'phone': list(set(phones)),
        'linkedin': list(set(re.findall(linkedin_pattern, text, re.IGNORECASE))),
        'github': list(set(re.findall(github_pattern, text, re.IGNORECASE)))
    }

def extract_summary(text: str) -> str:
    """Extract professional summary"""
    summary_keywords = ['summary', 'objective', 'profile', 'about', 'overview']

    lines = text.split('\n')
    for i, line in enumerate(lines):
        line_lower = line.lower().strip()
        if any(kw in line_lower for kw in summary_keywords) and len(line_lower) < 50:
            summary_lines = []
            for j in range(i+1, min(i+10, len(lines))):
                next_line = lines[j].strip()
                # Stop at next section
                if any(keyword in next_line.lower() for keyword in ['education', 'experience', 'technical skills', 'work']):
                    break
                if next_line and len(next_line) > 20:
                    summary_lines.append(next_line)
                    if len(' '.join(summary_lines)) > 300:
                        break
            return ' '.join(summary_lines)

    return ""

def extract_skills_optimized(text: str, similarity_threshold: float = 0.60) -> List[str]:
    """
    Extract skills using BOTH keyword matching and semantic similarity
    """
    found_skills = set()
    text_lower = text.lower()

    # Method 1: Direct keyword matching (fast and accurate)
    for skill in skills_list:
        skill_lower = skill.lower()
        pattern = r'\b' + re.escape(skill_lower) + r'\b'
        if re.search(pattern, text_lower):
            found_skills.add(skill)

    # Method 2: Semantic similarity for variations
    if len(found_skills) < 10:
        text_emb = sbert_model.encode(text, show_progress_bar=False)
        similarities = util.cos_sim(text_emb, skills_embeddings)[0]

        for i, score in enumerate(similarities):
            if score > similarity_threshold:
                found_skills.add(skills_list[i])

    return sorted(list(found_skills))

def extract_job_titles_optimized(text: str, similarity_threshold: float = 0.65) -> List[str]:
    """Extract job titles using keyword matching and similarity"""
    found_titles = set()
    text_lower = text.lower()

    for title in job_titles_list:
        title_lower = title.lower()
        if title_lower in text_lower:
            found_titles.add(title)

    if len(found_titles) < 5:
        text_emb = sbert_model.encode(text, show_progress_bar=False)
        similarities = util.cos_sim(text_emb, job_titles_embeddings)[0]

        for i, score in enumerate(similarities):
            if score > similarity_threshold:
                found_titles.add(job_titles_list[i])

    return sorted(list(found_titles))

def extract_education(text: str) -> List[Dict]:
    """Extract education information with improved parsing"""
    education = []

    lines = text.split('\n')
    in_education = False
    edu_lines = []

    for line in lines:
        line_stripped = line.strip()
        if 'education' in line_stripped.lower() and len(line_stripped) < 30:
            in_education = True
            continue

        if in_education:
            if any(keyword in line_stripped.lower() for keyword in ['experience', 'work', 'project', 'technical skills']) and len(line_stripped) < 40:
                break
            if line_stripped:
                edu_lines.append(line_stripped)

    if edu_lines:
        edu_text = ' '.join(edu_lines)
        doc = nlp(edu_text)

        # Extract organizations - filter out tech keywords
        all_orgs = [ent.text for ent in doc.ents
                    if ent.label_ == 'ORG' and ent.text.lower() not in TECH_KEYWORDS]

        # Prioritize universities over faculties
        university = ""
        faculty = ""

        for org in all_orgs:
            if 'university' in org.lower() or 'college' in org.lower() or 'institute' in org.lower():
                university = org
            elif 'faculty' in org.lower() or 'school of' in org.lower() or 'department' in org.lower():
                faculty = org

        # Use university as primary school, or faculty if no university found
        school = university if university else (faculty if faculty else (all_orgs[0] if all_orgs else ""))

        # Extract degrees and faculty information
        degree_patterns = [
            r'(Bachelor|Master|PhD|Ph\.D\.|B\.Sc\.|M\.Sc\.|B\.A\.|M\.A\.|Associate|Diploma)[^.]*?(?:in|of)\s+([\w\s]+?)(?:\s+\d{4}|$|,|\|)',
            r'(BS|MS|BA|MA|BSc|MSc)\s+(?:in\s+)?([\w\s]+?)(?:\s+\d{4}|$|,|\|)',
            r'Faculty of ([\w\s]+?)(?:\s+\d{4}|$|,|\s+and\s+)',
        ]

        degree_info = ""
        field_of_study = ""

        for pattern in degree_patterns:
            matches = re.findall(pattern, edu_text, re.IGNORECASE)
            if matches:
                if len(matches[0]) == 2:  # Degree type + field
                    degree_type, field = matches[0]
                    degree_info = f"{degree_type.strip()} - {field.strip()}"
                    field_of_study = field.strip()
                elif isinstance(matches[0], str):  # Just field (Faculty of X)
                    field_of_study = matches[0].strip()
                    degree_info = f"Bachelor - {field_of_study}"  # Assume Bachelor's
                break

        # If we found faculty but no degree, use faculty as degree
        if not degree_info and faculty:
            degree_info = faculty

        # Extract years
        years = re.findall(r'\b(19\d{2}|20\d{2})\b', edu_text)

        # Extract GPA
        gpa_match = re.search(r'GPA[:\s]*([\d.]+)', edu_text, re.IGNORECASE)
        gpa = gpa_match.group(1) if gpa_match else ""

        # Extract location
        location = ""
        location_patterns = [
            r'([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*),\s*([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*),\s*([A-Z]{2,})',  # City, Region, Country
            r'([A-Z][a-z]+),\s*([A-Z]{2,})',  # City, Country
        ]

        for pattern in location_patterns:
            loc_match = re.search(pattern, edu_text)
            if loc_match:
                location = loc_match.group(0)
                break

        # Extract GPE entities as fallback for location
        if not location:
            locations = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
            if locations:
                location = ', '.join(locations[:2])  # Take first 2 location entities

        if school or degree_info:
            education.append({
                'degree': degree_info,
                'school': school,
                'year': f"{years[0]} - {years[1]}" if len(years) >= 2 else years[0] if years else "",
                'gpa': gpa,
                'location': location
            })

    return education

def parse_duration(duration_str: str) -> Tuple[str, str]:
    """Parse duration string into start and end dates"""
    if not duration_str:
        return "", ""

    parts = re.split(r'[-–—]', duration_str)
    if len(parts) != 2:
        return "", ""

    start = parts[0].strip()
    end = parts[1].strip()

    return start, end

def calculate_duration(start_str: str, end_str: str) -> float:
    """Calculate duration in years"""
    try:
        if not start_str:
            return 0.0

        start = dateparser.parse(start_str)
        if not start:
            return 0.0

        if any(word in end_str.lower() for word in ['present', 'current', 'now']):
            end = datetime.now()
        else:
            end = dateparser.parse(end_str)
            if not end:
                return 0.0

        delta = end - start
        years = delta.days / 365.25
        return round(max(years, 0.0), 1)
    except:
        return 0.0

def extract_experience(text: str) -> List[Dict]:
    """Extract work experience with improved parsing"""
    experiences = []

    lines = text.split('\n')
    in_experience = False
    exp_lines = []

    for line in lines:
        line_stripped = line.strip()
        if any(kw in line_stripped.lower() for kw in ['work experience', 'experience', 'employment history']) and len(line_stripped) < 50:
            in_experience = True
            continue

        if in_experience:
            if any(keyword in line_stripped.lower() for keyword in ['education', 'project', 'technical skills', 'certification']) and len(line_stripped) < 40:
                break
            if line_stripped:
                exp_lines.append(line_stripped)

    if not exp_lines:
        return []

    # Split into entries - improved logic
    entries = []
    current_entry = []

    for line in exp_lines:
        line = line.strip()

        # New entry starts with a job title + date OR just date pattern
        is_new_entry = False

        # Check if line contains date range
        if re.search(duration_pattern, line, re.IGNORECASE):
            # This is likely a header line with job title + dates
            is_new_entry = True
        # Check if it's a standalone location/type line after we have content
        elif current_entry and re.search(r'^\s*(Remote|On-?site|Hybrid)\s*$', line, re.IGNORECASE):
            # Add to current entry, don't start new one
            current_entry.append(line)
            continue
        # Check for bullet points (description lines)
        elif line.startswith('–') or line.startswith('-') or line.startswith('•'):
            if current_entry:
                current_entry.append(line)
                continue

        if is_new_entry:
            if current_entry:
                entries.append("\n".join(current_entry))
            current_entry = [line]
        else:
            if current_entry or not line.startswith('–'):
                current_entry.append(line)

    if current_entry:
        entries.append("\n".join(current_entry))

    # Parse each entry
    for entry in entries:
        # Extract dates first
        duration_matches = re.findall(duration_pattern, entry, re.IGNORECASE)
        duration_str = ""
        if duration_matches:
            start, end = duration_matches[0]
            duration_str = f"{start} - {end}"

        # Extract the first line (usually contains title)
        first_line = entry.split('\n')[0]

        # Remove date from first line to get title
        title = re.sub(duration_pattern, '', first_line, flags=re.IGNORECASE).strip()
        title = re.sub(r'\s+', ' ', title)  # Clean extra spaces

        # Extract company using NER but filter tech keywords
        doc = nlp(entry)
        companies = [ent.text for ent in doc.ents
                    if ent.label_ == 'ORG' and
                    ent.text.lower() not in TECH_KEYWORDS ]
        # Extract company
#         companies = [ent.text for ent in doc.ents if ent.label_ == 'ORG']

        # Detect job type
        job_type = "Full-time"
        if re.search(r'\bIntern\b|\bInternship\b', entry, re.IGNORECASE):
            job_type = "Internship"
        elif re.search(r'\bRemote\b', entry, re.IGNORECASE):
            job_type = "Remote"
        elif re.search(r'\bContract\b|\bFreelance\b', entry, re.IGNORECASE):
            job_type = "Contract"
        elif re.search(r'\bPart-time\b', entry, re.IGNORECASE):
            job_type = "Part-time"

        # Extract location
        location = ""
        location_match = re.search(r'\b(Remote|On-?site|Hybrid)\b', entry, re.IGNORECASE)
        if location_match:
            location = location_match.group(1)

        # Calculate years
        start, end = parse_duration(duration_str)
        years = calculate_duration(start, end)

        if title or companies:
            experiences.append({
                'title': title,
                'company': companies[0] if companies else "",
                'duration': duration_str,
                'location': location,
                'job_type': job_type,
                'years': years
            })

    return experiences

Main CV Parsing Function

In [ ]:
def parse_cv(file_path: str) -> Optional[Dict]:
    """
    Main function to parse CV

    Args:
        file_path: Path to CV file

    Returns:
        Dictionary containing extracted data or None on error
    """
    try:
        logger.info(f"Starting to parse CV: {file_path}")

        # 1. Extract text
        text = extract_text(file_path)
        if not text or len(text.strip()) < 50:
            logger.error("Extracted text is too short or empty")
            return None

        logger.info(f"Extracted {len(text)} characters from CV")

        # 2. Clean text
        cleaned_text = clean_text(text)

        # 3. Extract all information
        logger.info("Extracting CV information...")

        cv_data = {
            "name": extract_name(text),
            **extract_contact_info(text),
            "summary": extract_summary(text),
            "education": extract_education(text),
            "experience": extract_experience(text),
            "skills": extract_skills_optimized(cleaned_text),
            "job_titles": extract_job_titles_optimized(cleaned_text),
            "total_experience_years": 0.0
        }

        # 4. Calculate total experience
        total_exp = sum(exp.get('years', 0.0) for exp in cv_data['experience'])
        cv_data['total_experience_years'] = round(total_exp, 1)

        # 5. Validation
        if not cv_data['name']:
            logger.warning("⚠️ Could not extract candidate name")
        if not cv_data['email']:
            logger.warning("⚠️ Could not extract email address")

        logger.info(f"✅ Successfully parsed CV:")
        logger.info(f"   - Skills: {len(cv_data['skills'])}")
        logger.info(f"   - Job Titles: {len(cv_data['job_titles'])}")
        logger.info(f"   - Experience Entries: {len(cv_data['experience'])}")
        logger.info(f"   - Education Entries: {len(cv_data['education'])}")

        return cv_data

    except Exception as e:
        logger.error(f"❌ Error parsing CV: {str(e)}", exc_info=True)
        return None

Validation & Output Functions

In [ ]:
def save_to_json(cv_data: Dict, output_path: str) -> bool:
    """Save data to JSON file"""
    try:
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(cv_data, f, indent=4, ensure_ascii=False)
        logger.info(f"✅ CV data saved to {output_path}")
        return True
    except Exception as e:
        logger.error(f"❌ Error saving to JSON: {str(e)}")
        return False

def print_cv_summary(cv_data: Dict):
    """Print summary of extracted data"""

In [ ]:
def main():
    """Example usage"""
    # Path to your CV file
    file_path = "/content/Amr Muarak CV.pdf"

    # Parse CV
    result = parse_cv(file_path)

    if result:
        # Print summary
        print_cv_summary(result)

        # Save results
        output_path = "/content/parsed_cv.json"
        save_to_json(result, output_path)

        # Print full JSON
        print("\n📄 Full JSON Output:")
        print(json.dumps(result, indent=2, ensure_ascii=False))
    else:
        print("❌ Failed to parse CV")

In [ ]:
main()


📄 Full JSON Output:
{
  "name": "Amr Ashraf Mubarak",
  "email": [
    "amrrdev@gmail.com"
  ],
  "phone": [
    "+20 120 456 2326"
  ],
  "linkedin": [
    "linkedin.com/in/amramubarak"
  ],
  "github": [
    "github.com/amrrdev"
  ],
  "summary": "",
  "education": [
    {
      "degree": "Bachelor - Science in Computer Science Oct",
      "school": "Benha University",
      "year": "2022 - 2026",
      "gpa": "3.6",
      "location": ""
    }
  ],
  "experience": [
    {
      "title": "Software Engineer – Linux Foundation Decentralized Trust – Remote",
      "company": "Software Engineer",
      "duration": "May 2025 - Jul 2025",
      "location": "Remote",
      "job_type": "Remote",
      "years": 0.2
    },
    {
      "title": "MEAN Stack Developer Intern – National Telecommunication Institute (NTI)",
      "company": "National Telecommunication Institute",
      "duration": "Jul 2025 - Aug 2025",
      "location": "",
      "job_type": "Internship",
      "years": 0.1
    }
 